# 🚀 SUDEPI — Full Potential Training Pipeline (NVIDIA A100-80GB + 167GB RAM)

Notebook ini dioptimalkan khusus untuk melatih model deteksi uang Rupiah **SUDEPI** pada Google Colab Pro dengan GPU **NVIDIA A100-SXM4-80GB** dan **167.1 GB RAM Sistem**.

> **Fitur Full Potential yang Aktif:**
> - `cache='ram'`: Seluruh 44.376 citra dimuat ke RAM 167 GB (0% disk I/O, akselerasi murni).
> - `workers=12`: Paralelisasi dataloader tingkat tinggi.
> - `mixup=0.15` + `copy_paste=0.1`: Simulasi uang bertumpuk dan koin di atas uang kertas.
> - `epochs=100` (`patience=25`): Akurasi maksimal konvergen (~6–9 menit di A100-80GB).
>
> **Kontrak Sistem Terkunci Rapat (ADR-0001, ADR-0002, ADR-0007, ADR-0009):**
> - **8 Kelas Resmi**: `0: rp1000` s.d. `7: koin` (urutan persis sesuai `TABEL_DENOMINASI` di `src/contracts/uang.ts`).
> - **`imgsz = 320`**: Resolusi hemat komputasi CPU HP (Samsung Galaxy M32).
> - **`nms = False`**: NMS class-agnostic sudah ditangani Farrel di TypeScript.
> - **Bentuk Tensor**: Output ONNX wajib tepat `[1, 12, 2100]` (12 kanal = 4 bbox + 8 kelas, 2100 jangkar).

### 1. Pasang Dependensi & Inisialisasi Monster GPU (A100-80GB TF32)

In [ ]:
!nvidia-smi
!pip install -q ultralytics onnx onnxruntime pyyaml

import torch
print(f"CUDA Tersedia: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🔥 Akselerator Aktif : {gpu_name}")
    print(f"💾 Kapasitas VRAM    : {vram_gb:.1f} GB")
    
    # Aktifkan akselerasi TensorFloat-32 (TF32) untuk Tensor Cores Generasi ke-3 A100
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("🚀 TensorFloat-32 (TF32) diaktifkan untuk throughput maksimal!")

### 2. Ekstrak Dataset Gabungan SUDEPI (8 Kelas)

**Cara pakai:**
1. Klik ikon folder di sidebar kiri Colab.
2. Tarik (*drag and drop*) berkas `dataset_delta_sudepi.zip` dari komputermu ke sidebar Colab.
3. Jalankan sel di bawah ini.

In [ ]:
import os, zipfile, re, shutil

# Cari berkas dataset zip
zip_candidates = [
    'dataset_delta_sudepi.zip', 'dataset_sudepi.zip',
    '/content/dataset_delta_sudepi.zip', '/content/dataset_sudepi.zip',
    '/content/drive/MyDrive/dataset_delta_sudepi.zip', '/content/drive/MyDrive/dataset_sudepi.zip'
]
zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

if not zip_path:
    raise FileNotFoundError("⚠️ Berkas zip dataset belum ditemukan! Silakan drag & drop dataset_delta_sudepi.zip ke sidebar kiri Colab.")

print(f"Mengekstrak {zip_path}...")
os.makedirs('dataset', exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('dataset')

# Sesuaikan path di data.yaml agar menunjuk ke direktori absolut /content/dataset
yaml_path = 'dataset/data.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    content = f.read()

content = re.sub(r'path:.*', 'path: /content/dataset', content)
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(content)

print("✅ Ekstraksi selesai dan path: /content/dataset siap!")
with open(yaml_path, 'r') as f:
    print(f.read())

### 3. Pelatihan YOLOv8n (FULL POTENTIAL: A100-80GB + RAM 167GB)
- `cache='ram'` (Semua gambar disimpan di RAM 167 GB, 0% disk latency)
- `batch=128`, `workers=12`
- `mixup=0.15`, `copy_paste=0.1` (Uang bertumpuk & koin di atas uang kertas)
- Estimasi waktu: **~6–9 menit** untuk 100 epoch.

In [ ]:
from ultralytics import YOLO

# Muat bobot dasar resmi YOLOv8-Nano
model = YOLO('yolov8n.pt')

# Mulai pelatihan FULL POTENTIAL
results = model.train(
    data='dataset/data.yaml',
    imgsz=320,          # WAJIB 320 (ADR-0001, ADR-0009)
    epochs=100,         # FULL POTENTIAL: 100 epoch untuk akurasi maksimal
    batch=128,          # Optimal untuk A100 80GB VRAM
    workers=12,         # Maksimal vCPU server Colab High-RAM
    cache='ram',        # KUNCI UTAMA: Caching 100% citra di RAM 167 GB
    patience=25,        # Early stopping otomatis jika mAP sudah puncak
    # --- Augmentasi Realistis Lapak Pasar & Tunanetra ---
    degrees=15.0,       # Uang dipegang miring
    shear=5.0,
    perspective=0.0005, # Sudut pandang kamera ponsel
    hsv_v=0.5,          # Simulasi lapak pasar temaram/redup
    hsv_s=0.7,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=15,
    mixup=0.15,         # Simulasi lembaran bertumpuk/berdempetan
    copy_paste=0.1      # Tempelan koin di atas uang kertas
)

print("🎉 Pelatihan Full Potential selesai!")

### 4. Ekspor ke Format ONNX (Wajib `nms=False`)
NMS ditangani oleh decoder TypeScript Farrel di HP, jadi ekspor WAJIB mematikan NMS bawaan.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import shutil

# Cari bobot terbaik hasil training
save_dir = Path(results.save_dir) if hasattr(results, 'save_dir') else Path('runs/detect/train')
best_pt = save_dir / 'weights' / 'best.pt'
print(f"Mengekspor dari: {best_pt}")

model_best = YOLO(str(best_pt))
exported_path = model_best.export(
    format='onnx',
    imgsz=320,
    opset=12,
    simplify=True,
    nms=False,          # WAJIB False (ADR-0002)
    dynamic=False
)

# Salin ke nama resmi sudepi.onnx
shutil.copy2(exported_path, 'sudepi.onnx')
print("✅ sudepi.onnx berhasil dibuat dan siap diverifikasi!")

### 5. Verifikasi Keselarasan Tensor Output
Output WAJIB `[1, 12, 2100]` (12 kanal = 4 koordinat + 8 kelas, 2100 jangkar).

In [ ]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession('sudepi.onnx')
inp_node = session.get_inputs()[0]
out_node = session.get_outputs()[0]

print(f"Node Masukan : nama='{inp_node.name}', bentuk={inp_node.shape}")
print(f"Node Keluaran: nama='{out_node.name}', bentuk={out_node.shape}")

shape = out_node.shape
if shape[1] == 12 and shape[2] == 2100:
    print("\n=======================================================")
    print("🎉 VERIFIKASI SUKSES: Model 100% cocok dengan kode Farrel!")
    print("=======================================================")
else:
    print(f"\n❌ GALAT FATAL: Bentuk output {shape} tidak cocok dengan [1, 12, 2100]!")

### 6. (Opsional) Uji Gerbang Mutu Kuantisasi INT8
Sesuai aturan Farrel di `model/ekspor.py`:
> Ukur mAP@0.5 model INT8 terhadap FP32 pada data validasi.
> Jika mAP turun > 3.0 poin, INT8 DITOLAK dan kita tetap memakai FP32.

In [ ]:
try:
    from onnxruntime.quantization import QuantType, quantize_dynamic
    
    print("Mengukur mAP FP32...")
    val_fp32 = model_best.val(data='dataset/data.yaml', imgsz=320, verbose=False)
    map_fp32 = float(val_fp32.box.map50) * 100
    
    print("Membuat sudepi-int8.onnx...")
    quantize_dynamic('sudepi.onnx', 'sudepi-int8.onnx', weight_type=QuantType.QUInt8)
    
    print("Mengukur mAP INT8...")
    val_int8 = YOLO('sudepi-int8.onnx').val(data='dataset/data.yaml', imgsz=320, verbose=False)
    map_int8 = float(val_int8.box.map50) * 100
    
    drop = map_fp32 - map_int8
    print(f"\nFP32 mAP@0.5: {map_fp32:.2f}%")
    print(f"INT8 mAP@0.5: {map_int8:.2f}% (Turun {drop:.2f} poin)")
    
    if drop <= 3.0:
        print("✅ INT8 DITERIMA! Mengganti sudepi.onnx dengan versi INT8.")
        shutil.copy2('sudepi-int8.onnx', 'sudepi.onnx')
    else:
        print("⚠️ INT8 DITOLAK: Penurunan mAP > 3 poin. Tetap menggunakan FP32.")
except Exception as e:
    print(f"Kuantisasi INT8 dilewati/galat: {e}")

### 7. Unduh Model untuk Farrel
Unduh berkas `sudepi.onnx` dan letakkan di `F:\Code\public\model\sudepi.onnx`.

In [ ]:
from google.colab import files
files.download('sudepi.onnx')